In [1]:
# Preprocessing — Text pipeline (URLs, stopwords, NLTK lemmatization)
# Install / refresh deps if needed (matches project requirements.txt)
%pip install -q pandas nltk tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name in {"notebooks", "notebook_runs"} else NOTEBOOK_DIR
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from text_preprocessing import ensure_nltk_resources, preprocess_text

ensure_nltk_resources()

import pandas as pd
from tqdm.auto import tqdm

tqdm.pandas()
print("Project root:", ROOT)

Project root: D:\ml project\Social media and sentiment analysis


In [3]:
DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

INPUT_CSV = DATA_DIR / "sample_dataset.csv"
OUTPUT_CSV = DATA_DIR / "processed_dataset.csv"

df = pd.read_csv(INPUT_CSV)
print("Rows:", len(df))
print(df.columns.tolist())

Rows: 4500
['text', 'sentiment', 'target']


In [4]:
# Full pipeline: remove URLs → lowercase → tokenize → drop English stopwords → lemmatize (WordNet)
# use_pos_tag=True uses NLTK POS tags for lemmas (better quality, slower). Set False for very large CSVs.
USE_POS_TAG = True

df["text_clean"] = df["text"].progress_apply(
    lambda t: preprocess_text(str(t), use_pos_tag=USE_POS_TAG)
)

  0%|          | 0/4500 [00:00<?, ?it/s]

In [5]:
sample = df[["text", "text_clean", "sentiment"]].head(12)
print(sample.to_string(index=False))

                                text           text_clean sentiment
            standard procedure #1394   standard procedure   neutral
              terrible quality #7132     terrible quality  negative
          will follow up later #5469         follow later   neutral
          noted for the record #7456          note record   neutral
              highly recommend #5189     highly recommend  positive
              hate this update #4329          hate update  negative
       so frustrated right now #3274      frustrate right  negative
          noted for the record #5824          note record   neutral
           loving this so much #5383            love much  positive
                 this is awful #2478                awful  negative
totally disappointed with this #8008 totally disappointed  negative
           would not recommend #4510      would recommend  negative


In [6]:
export = df[["text", "text_clean", "sentiment", "target"]].copy()
export.to_csv(OUTPUT_CSV, index=False)
print("Wrote:", OUTPUT_CSV)

Wrote: D:\ml project\Social media and sentiment analysis\data\processed_dataset.csv
